# Çizge Topolojisi

**Modül:** `pytop.graph_topology`  
**Kaynak:** Adams & Franzosa *Introduction to Topology* §13.1–13.4

---

Bir **topolojik çizge** (graf), köşelerin (vertex) nokta, kenarların (edge) kapalı aralık olarak modellendiği 1-boyutlu bir hücre kompleksidir.  
Topoloji, çizimdeki geometrik koordinatları değil, bağlantı yapısını — hangi vertex'lerin hangi edge'lerle bağlandığını — inceler.

**Motivasyon:**  
- Euler karakteristiği, kompaktlık ve bağlantılılığı tek bir sayıyla kodlar.
- Planarity (düzlemsellik), bir çizgenin kesişimsiz çizilip çizilemeyeceğini belirler.
- Kimyasal moleküller, çizge topolojisiyle bağlanma yapısı olarak temsil edilir.

Bu notebook beş bölümden oluşur:
1. **Konu** — Topolojik çizge tanımı, Euler karakteristiği, gömülme kavramı
2. **Teoremler** — Euler formülü, Kuratowski teoremi, planarity eşitsizliği
3. **API** — `pytop.graph_topology` fonksiyon referansı
4. **Örnekler** — Dört senaryo: temel çizgelerden kimyasal graflarına
5. **Alıştırmalar** — Kodlama ve teori

## 1. Konu

### 1.1 Topolojik Çizge Modeli

**Tanım.** Bir **topolojik çizge** $G = (V, E)$:
- $V$ — sonlu, boş olmayan, ayrık nokta kümesi (köşeler),
- $E$ — sonlu, birbirini iç kısımda kesmeyen kapalı aralık kümesi (kenarlar); her aralığın uç noktaları $V$'ye aittir.

Bu model, standart kombinatoryal çizge tanımının topolojik gerçekleştirmesidir.

### 1.2 Euler Karakteristiği

Topolojik bir çizge $G$ için **Euler karakteristiği**:
$$\chi(G) = V - E$$
burada $V = |\text{vertex kümesi}|$, $E = |\text{edge kümesi}|$.

| Çizge | $V$ | $E$ | $\chi$ |
|-------|-----|-----|--------|
| Aralık (tek edge) | 2 | 1 | $1$ |
| Döngü ($n$-çokgen) | $n$ | $n$ | $0$ |
| Theta çizgesi | 2 | 3 | $-1$ |
| İki bileşenli aralık | 4 | 2 | $2$ |

### 1.3 Gömülme ve Yüz Sayısı

Bir çizge **düzlemsel** (planar) ise düzleme kenarlar kesişmeden gömülebilir.  
Düzlemsel gömülmede **yüz** (face) sayısı $F$, Euler formülüyle ilişkilidir:
$$V - E + F = 1 + C$$
burada $C$ = bağlantılı bileşen sayısı. Bağlantılı çizgeler için $V - E + F = 2$.

In [ ]:
from pytop.graph_topology import (
    get_topological_graph_profiles,
    get_graph_embedding_profiles,
    get_graph_planarity_profiles,
    get_chemical_graph_profiles,
    get_graph_crossing_thickness_profiles,
    graph_euler_characteristic_summary,
    graph_embedding_ambient_summary,
    graph_planarity_summary,
    chemical_graph_molecule_summary,
    graph_topology_profile_registry,
)

print("=== Topolojik Çizge Profilleri ===")
for p in get_topological_graph_profiles():
    print(f"  {p.display_name}")
    print(f"    V={p.vertex_count}  E={p.edge_count}  C={p.component_count}  χ={p.euler_characteristic}")
    print(f"    Model: {p.graph_model}")
    print()

## 2. Teoremler

### Teorem 1 — Euler Formülü (Düzlemsel Çizgeler)

> **Teorem.** $G$ bağlantılı, düzlemsel bir çizge ve $F$ = düzlemsel gömülmedeki yüz sayısı ise:
> $$V - E + F = 2.$$

**Kanıt fikri (ağaç üzerinden endüksiyon).**  
- $G$ bir ağaç (tree) ise $E = V-1$, $F = 1$ (tek dış yüz); $V - (V-1) + 1 = 2$. $\checkmark$  
- Her adımda bir iç kenar silindiğinde $E$ bir azalır, komşu iki yüz birleşir ($F$ bir azalır); toplam değişmez. $\square$

---

### Teorem 2 — Planarity Eşitsizliği

> **Teorem.** $G$ basit, bağlantılı, düzlemsel çizge ve $V \geq 3$ ise:
> $$E \leq 3V - 6.$$

**Kanıt.** Her yüz en az 3 kenarla sınırlıdır; her kenar en fazla 2 yüzle komşudur, yani $3F \leq 2E$.  
Euler formülüne ($F = 2 - V + E$) koyulursa $3(2-V+E) \leq 2E$, yani $E \leq 3V-6$. $\square$

**Sonuç:** $K_5$: $V=5$, $E=10 > 3\cdot5-6 = 9$ → $K_5$ düzlemsel değil.

---

### Teorem 3 — Kuratowski Teoremi

> **Teorem.** Bir çizge $G$ düzlemsel olup olmadığı şu koşulla karakterize edilir:  
> $G$ düzlemsel **değildir** $\Longleftrightarrow$ $G$, $K_5$ veya $K_{3,3}$'ün bir subdivizyon (bölünmüş alt çizge) içerir.

**Not:**  
- $K_5$: 5 köşeli tam çizge — $V=5$, $E=10$.
- $K_{3,3}$: iki taraflı tam çizge (3+3) — $V=6$, $E=9$.
- "Subdivizyon" = kenara yeni köşe(ler) eklenerek yolu uzatmak.

---

### Teorem 4 — Bileşen Genellemesi

> **Teorem.** $G$, $C$ bağlantılı bileşene sahip bir çizge ise düzlemsel gömülmede:
> $$V - E + F = C + 1.$$

**Kanıt.** Her bileşene Teorem 1 uygulanır ve yüzler sayılır; dış yüz tek kez sayıldığından toplam $C + 1$. $\square$

In [ ]:
# Teorem 2 doğrulaması: E ≤ 3V - 6
print("Planarity eşitsizliği E ≤ 3V-6 kontrolü:")
for p in get_topological_graph_profiles():
    bound = 3 * p.vertex_count - 6
    ok = p.edge_count <= bound
    print(f"  {p.display_name:30s}  E={p.edge_count}  3V-6={bound}  {'✓' if ok else '✗'}")

print()
# K5 ve K3,3 sayısal kontrolü
for name, V, E in [("K5", 5, 10), ("K3,3", 6, 9)]:
    bound = 3 * V - 6
    print(f"  {name}: E={E}  3V-6={bound}  {'E ≤ 3V-6 sağlanıyor' if E <= bound else 'E > 3V-6 → DÜZLEMSELDEĞİL'}")

## 3. API Referansı

| Fonksiyon / Sınıf | Açıklama | Döndürür |
|-------------------|----------|----------|
| `TopologicalGraphProfile` | Temel topolojik çizge modeli (V, E, bileşen, χ) | `dataclass(frozen=True)` |
| `get_topological_graph_profiles()` | Tüm temel çizge profillerini döndürür | `tuple[TopologicalGraphProfile, ...]` |
| `graph_euler_characteristic_summary()` | Anahtar → Euler karakteristiği eşlemesi | `dict[str, int]` |
| `GraphEmbeddingProfile` | Çizgenin yüzey/düzleme gömülme profili | `dataclass(frozen=True)` |
| `get_graph_embedding_profiles()` | Tüm gömülme profillerini döndürür | `tuple[GraphEmbeddingProfile, ...]` |
| `graph_embedding_ambient_summary()` | Ortam uzayı → anahtar listesi eşlemesi | `dict[str, list[str]]` |
| `GraphPlanarityProfile` | Planarity + Kuratowski engel profili | `dataclass(frozen=True)` |
| `get_graph_planarity_profiles()` | Tüm planarity profillerini döndürür | `tuple[GraphPlanarityProfile, ...]` |
| `graph_planarity_summary()` | Anahtar → `is_planar` bool eşlemesi | `dict[str, bool]` |
| `ChemicalGraphProfile` | Kimyasal çizge modeli (atom=vertex, bağ=edge) | `dataclass(frozen=True)` |
| `get_chemical_graph_profiles()` | Tüm kimyasal çizge profillerini döndürür | `tuple[ChemicalGraphProfile, ...]` |
| `chemical_graph_molecule_summary()` | Anahtar → molekül modeli eşlemesi | `dict[str, str]` |
| `GraphCrossingThicknessProfile` | Geçiş sayısı ve kalınlık sinyali profili | `dataclass(frozen=True)` |
| `get_graph_crossing_thickness_profiles()` | Tüm geçiş/kalınlık profillerini döndürür | `tuple[GraphCrossingThicknessProfile, ...]` |
| `graph_topology_profile_registry()` | Kategori başına profil sayısı | `dict[str, int]` |

In [ ]:
registry = graph_topology_profile_registry()
print("Kayıt defteri özeti:")
total = 0
for k, v in registry.items():
    print(f"  {k:<40s}: {v}")
    total += v
print(f"  {'Toplam':<40s}: {total}")

## 4. Örnekler

In [ ]:
# Örnek 1: Euler karakteristiğini pozitif / sıfır / negatif olarak grupla
profiles = get_topological_graph_profiles()

pos  = [p for p in profiles if p.euler_characteristic > 0]
zero = [p for p in profiles if p.euler_characteristic == 0]
neg  = [p for p in profiles if p.euler_characteristic < 0]

print("χ > 0  (ağaç benzeri veya çok bileşenli):")
for p in pos:
    print(f"  χ={p.euler_characteristic:+d}  {p.display_name}  (C={p.component_count})")

print("\nχ = 0  (tek döngü):")
for p in zero:
    print(f"  χ={p.euler_characteristic:+d}  {p.display_name}")

print("\nχ < 0  (çok döngülü):")
for p in neg:
    print(f"  χ={p.euler_characteristic:+d}  {p.display_name}")

# Özet tablosu
print("\n--- Euler Karakteristiği Özeti ---")
for k, chi in graph_euler_characteristic_summary().items():
    print(f"  {k}: χ={chi}")

In [ ]:
# Örnek 2: Gömülme profillerini ortam uzayına göre grupla ve yüz sayısını göster
emb_profiles = get_graph_embedding_profiles()

print("=== Çizge Gömülme Profilleri ===")
for p in emb_profiles:
    faces = str(p.face_count) if p.face_count is not None else "tanımsız (yüz sayımı yok)"
    print(f"  {p.display_name}")
    print(f"    Ortam uzayı : {p.ambient_space}")
    print(f"    Yüz sayısı  : {faces}")
    print(f"    Sinyal      : {p.embedding_signal}")
    print()

# Ortam uzayı gruplama
ambient_summary = graph_embedding_ambient_summary()
print("Ortam uzayı → anahtar listesi:")
for space, keys in ambient_summary.items():
    print(f"  {space}: {keys}")

In [ ]:
# Örnek 3: Planarity profillerini Kuratowski engeli ile analiz et
plan_profiles = get_graph_planarity_profiles()

print("=== Planarity Analizi ===")
planar     = [p for p in plan_profiles if p.is_planar]
nonplanar  = [p for p in plan_profiles if not p.is_planar]

print("Düzlemsel çizgeler:")
for p in planar:
    print(f"  ✓ {p.display_name}  V={p.vertex_count}  E={p.edge_count}")
    print(f"    Engel: {p.obstruction_signal}")

print("\nDüzlemsel OLMAYAN çizgeler (Kuratowski engeli):")
for p in nonplanar:
    print(f"  ✗ {p.display_name}  V={p.vertex_count}  E={p.edge_count}")
    print(f"    Engel: {p.obstruction_signal}")

print("\n--- Planarity Özeti ---")
for k, v in graph_planarity_summary().items():
    print(f"  {k}: {'düzlemsel' if v else 'DÜZLEMSELDEĞİL'}")

In [ ]:
# Örnek 4: Kimyasal çizgeler ve geçiş sayısı uyarıları
chem_profiles  = get_chemical_graph_profiles()
cross_profiles = get_graph_crossing_thickness_profiles()

print("=== Kimyasal Çizge Profilleri ===")
for p in chem_profiles:
    print(f"  {p.display_name}")
    print(f"    Molekül modeli  : {p.molecule_model}")
    print(f"    Atom kuralı     : {p.atom_vertex_rule}")
    print(f"    Bağ kuralı      : {p.bond_edge_rule}")
    print(f"    Topoloji sinyali: {p.topology_signal}")
    print()

print("=== Geçiş Sayısı ve Kalınlık Uyarıları ===")
for p in cross_profiles:
    print(f"  {p.display_name}")
    print(f"    Geçiş sinyali  : {p.crossing_number_signal}")
    print(f"    Kalınlık sinyali: {p.thickness_signal}")
    print(f"    Kimya bağlamı  : {p.chemistry_context}")
    print()

print("Molekül modeli özeti:")
for k, mol in chemical_graph_molecule_summary().items():
    print(f"  {k}: {mol}")

## 5. Alıştırmalar

**Alıştırma 1.**  
`get_topological_graph_profiles()` profillerini kullanarak:
(a) `graph_euler_characteristic_summary()` çıktısını elle bir sözlük olarak yeniden oluşturun; `V - E` farkını doğrudan hesaplayarak profil verisindeki `euler_characteristic` ile karşılaştırın.  
(b) Bağlantılı bir çizge için $\chi(G) = V - E = 1 - \text{döngü sayısı}$ formülünü `theta_graph` profili üzerinde doğrulayın (theta çizgesinin kaç bağımsız döngüsü var?).  
(c) İki bileşenli aralık profilini ($C=2$) kullanarak $\chi = V - E = C$ eşitliğinin ağaç benzeri bileşenler için neden geçerli olduğunu açıklayın.

**Alıştırma 2.**  
`get_graph_embedding_profiles()` profillerini inceleyerek:
(a) `cycle_graph_plane_embedding` için $V - E + F = 2$ formülünü $V=4$, $E=4$, $F=2$ değerleriyle doğrulayın.  
(b) `theta_graph_plane_embedding` için $V=2$, $E=3$, $F=3$ değerleriyle Euler formülünü kontrol edin.  
(c) `interval_graph_line_embedding` profilinin `face_count=None` olmasının matematiksel nedenini açıklayın: doğruya gömülen bir yay hangi yüzleri oluşturur?

**Alıştırma 3.**  
`get_graph_planarity_profiles()` profillerini kullanarak:
(a) $K_5$ için planarity eşitsizliğini $E \leq 3V-6$ ile sayısal olarak kontrol edin ve neden ihlal edildiğini gösterin.  
(b) $K_{3,3}$ için ise bipartite graflara özgü daha sıkı sınır $E \leq 2V-4$'ü kontrol edin.  
(c) `graph_planarity_summary()` çıktısını kullanarak düzlemsel ve düzlemsel olmayan profilleri ayırın; Kuratowski teoreminin bu iki engel dışında başka bir engel tanımlamadığını `obstruction_signal` alanından okuyarak açıklayın.

**Alıştırma 4.**  
`get_chemical_graph_profiles()` ve `get_graph_crossing_thickness_profiles()` profillerini birlikte analiz edin:
(a) Benzen halkasının altıgen döngü çizgesine (`cycle_graph_circle` profiliyle karşılaştırın) neden düzlemsel olduğunu `topology_signal` alanından alıntı yaparak açıklayın.  
(b) `molecular_bond_graph_crossing_warning` profilinin `teaching_note` alanını okuyun: çizimde görünen bir geçiş (crossing) neden her zaman bir kimyasal bağ kesişimi anlamına gelmez?  
(c) `graph_topology_profile_registry()` ile toplam profil sayısını yazdırın ve her kategorinin hangi teoremle (Euler formülü, Kuratowski, planarity eşitsizliği) ilişkili olduğunu eşleştirin.

In [ ]:
# Alıştırma 4c kurulumu — registry ve kategori-teorem eşleştirmesi
registry = graph_topology_profile_registry()

teorem_eslestirme = {
    "topological_graph_profiles"       : "Euler karakteristiği χ = V - E",
    "graph_embedding_profiles"         : "Euler formülü V - E + F = 2",
    "graph_planarity_profiles"         : "Kuratowski teoremi (K5 / K3,3 engeli)",
    "chemical_graph_profiles"          : "Çizge izomorfizmi → kimyasal izomer ayrımı",
    "graph_crossing_thickness_profiles": "Geçiş sayısı / kalınlık (planarity ötesi)",
}

print(f"{'Kategori':<40} {'Profil':>6}  Teorem")
print("─" * 95)
total = 0
for k, v in registry.items():
    teorem = teorem_eslestirme.get(k, "—")
    print(f"  {k:<38} {v:>6}  {teorem}")
    total += v
print("─" * 95)
print(f"  {'Toplam':<38} {total:>6}")

---

## Özet

| Kavram | Ana Sonuç |
|--------|-----------|
| Topolojik çizge | Köşeler = nokta, kenarlar = kapalı aralık; koordinat değil bağlantı yapısı |
| Euler karakteristiği | $\chi = V - E$; homotopi tipi invariantı |
| Euler formülü | Bağlantılı düzlemsel çizgede $V - E + F = 2$ |
| Planarity eşitsizliği | $E \leq 3V-6$ (basit, bağlantılı, düzlemsel; $V \geq 3$) |
| Kuratowski teoremi | $G$ düzlemsel değil $\Leftrightarrow$ $K_5$ veya $K_{3,3}$ subdivizyonu içerir |
| Kimyasal çizge | Atom = vertex, bağ = edge; bağlanma topolojisi izomorfizm altında incelenir |
| Geçiş sayısı | Düzlemsel olmayan çizgedeki en az geçiş sayısı; kalınlık = gereken düzlem katman sayısı |

**Bir sonraki adım:** `pytop.fixed_point_profiles` — Brouwer, Schauder, Banach sabit nokta teoremleri.